# FewShotPromptTemplate — 예제로 LLM을 가르치는 퓨샷 프롬프팅

Ch01에서 LLM 호출을 배웠다면, 이제 프롬프트를 체계적으로 관리하는 법을 배워요.
특히 이번 노트북에서는 LLM에게 **예제(example)**를 직접 보여주어 출력 형식과 추론 패턴을 학습시키는 퓨샷(Few-Shot) 기법을 다뤄요.

"설명"보다 "예시"가 더 효과적일 때가 있어요 — LLM도 마찬가지예요.

## 학습 목표

- `FewShotPromptTemplate`으로 예제를 포함한 프롬프트를 만들 수 있어요
- `SemanticSimilarityExampleSelector`(의미 유사도 예시 선택기)로 입력에 맞는 예제를 자동 선택할 수 있어요
- `FewShotChatMessagePromptTemplate`로 대화형 퓨샷 프롬프트를 구성할 수 있어요
- `BaseExampleSelector`를 상속해 커스텀 예시 선택기를 구현할 수 있어요

## 사전 지식

- `01-PromptTemplate` 노트북의 `PromptTemplate` 기본 사용법
- LCEL 파이프라인 (`|` 연산자) 기초

---

> **FewShot 패턴 구조도** — 아래 다이어그램은 예제가 선택·포맷·조립되어 최종 프롬프트가 되는 흐름을 보여줘요.

```mermaid
flowchart LR
    EX["예제 풀<br/>examples 리스트<br/>(질문 + 정답 쌍)"]:::storage
    SEL["ExampleSelector<br/>의미 유사도 계산으로<br/>관련 예제 k개 선택"]:::process
    FMT["example_prompt<br/>각 예제를<br/>문자열로 포맷"]:::process
    ASM["FewShotPromptTemplate<br/>prefix + 예제들 + suffix<br/>순서로 조립"]:::process
    FINAL["완성된 프롬프트<br/>(예제 포함)"]:::output
    LLM["LLM"]:::external
    ANS["응답<br/>(예제 패턴 따름)"]:::output

    EX -->|"모든 예제 전달"| SEL
    SEL -->|"선택된 예제"| FMT
    FMT -->|"포맷된 예제 문자열"| ASM
    ASM --> FINAL
    FINAL --> LLM
    LLM --> ANS

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

In [ ]:
# 필수 라이브러리 import
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate, FewShotChatMessagePromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# 모델 초기화
# temperature=0: 일관된 출력을 위해 설정
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. FewShotPromptTemplate — 예제를 프롬프트에 담기

`FewShotPromptTemplate`(퓨샷 프롬프트 템플릿)은 여러 예제를 프롬프트에 포함시켜 LLM이 패턴을 학습하도록 돕는 템플릿이에요.

> **실무 팁:** Chain-of-Thought(사고 연쇄, CoT) 방식처럼 단계별 추론 예제를 넣으면, 복잡한 질문에도 LLM이 같은 방식으로 단계를 밟아 답해요.

### 구성 요소 요약

| 파라미터 | 역할 |
|---------|------|
| `examples` | 예제 딕셔너리 리스트 |
| `example_prompt` | 각 예제를 어떻게 문자열로 변환할지 정의 |
| `suffix` | 예제 뒤에 붙는 실제 질문 템플릿 |
| `input_variables` | `suffix`에서 사용하는 변수명 리스트 |

### 1.1 기본 사용법 - Chain-of-Thought 예제

복잡한 추론이 필요한 질문에 단계별로 생각하는 패턴을 학습시킵니다.


In [2]:
# 시나리오: 복잡한 질문에 대해 단계별 추론(Chain-of-Thought)을 수행하도록 학습

# 1단계: Few-Shot 예제 정의
#        - 각 예제는 question과 answer로 구성
#        - answer는 단계별 추론 과정을 포함
examples = [
    {
        "question": "아이폰과 갤럭시 중 어느 것이 먼저 출시되었나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 최초의 아이폰은 언제 출시되었나요?
중간 답변: 최초의 아이폰은 2007년 6월 29일에 출시되었습니다.
추가 질문: 최초의 갤럭시는 언제 출시되었나요?
중간 답변: 최초의 갤럭시(갤럭시 S)는 2010년 6월 4일에 출시되었습니다.
최종 답변은: 아이폰""",
    },
    {
        "question": "테슬라의 CEO는 몇 년생인가요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: 테슬라의 CEO는 누구인가요?
중간 답변: 테슬라의 CEO는 일론 머스크입니다.
추가 질문: 일론 머스크는 언제 태어났나요?
중간 답변: 일론 머스크는 1971년 6월 28일에 태어났습니다.
최종 답변은: 1971년생""",
    },
    {
        "question": "ChatGPT와 Bard는 같은 회사에서 만들었나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
추가 질문: ChatGPT는 어느 회사에서 만들었나요?
중간 답변: ChatGPT는 OpenAI에서 만들었습니다.
추가 질문: Bard는 어느 회사에서 만들었나요?
중간 답변: Bard는 Google에서 만들었습니다.
추가 질문: OpenAI와 Google은 같은 회사인가요?
중간 답변: 아니요, OpenAI와 Google은 다른 회사입니다.
최종 답변은: 아니요""",
    },
]

print("=" * 60)
print("📚 준비된 예제 개수:", len(examples))
print("=" * 60)


📚 준비된 예제 개수: 3


In [ ]:
# ---------------------------------------------------
# 2단계: 예제 포맷 정의 및 포맷 결과 확인
# ---------------------------------------------------

# 2단계: 각 예제를 어떻게 포맷할지 정의
# PromptTemplate: question과 answer 변수를 담은 단일 예제 포맷


# 예제가 어떻게 포맷되는지 확인



In [ ]:
# ---------------------------------------------------
# 3단계: FewShotPromptTemplate 생성 및 최종 프롬프트 확인
# ---------------------------------------------------

# 3단계: FewShotPromptTemplate 생성
# FewShotPromptTemplate: examples + example_prompt + suffix를 조합하여
#                         모든 예제가 포함된 완성 프롬프트 생성


# 최종 프롬프트 확인
question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"



In [ ]:
# ---------------------------------------------------
# 4단계: 체인 구성 및 실행 - Chain-of-Thought 응답 확인
# ---------------------------------------------------

# 4단계: 체인 구성 및 실행
# 파이프(|) 연산자: 왼쪽 출력이 오른쪽 입력으로 자동 전달



## 2. Example Selector — 입력에 맞는 예제를 자동으로 고르기

예제가 많아질수록 전부 프롬프트에 넣으면 토큰 비용이 커져요. **Example Selector**(예시 선택기)는 입력과 가장 관련 있는 예제만 골라줘요.

> **주의:** `SemanticSimilarityExampleSelector`는 내부적으로 **벡터 DB**(여기서는 Chroma)를 사용해요. 예제를 임베딩해 저장하는 시간이 걸리므로, 예제 수가 아주 많으면 초기화 시간이 길어질 수 있어요.

### 주요 Selector 종류

| Selector | 선택 기준 | 적합한 상황 |
|---------|----------|------------|
| `SemanticSimilarityExampleSelector` | 의미 유사도 | 입력과 의미가 비슷한 예제 선택 |
| `MaxMarginalRelevanceExampleSelector` | 유사도 + 다양성 | 중복 예제 없이 다양하게 선택 |
| `LengthBasedExampleSelector` | 프롬프트 길이 | 토큰 한도 내에서 최대한 예제 포함 |

### 2.1 SemanticSimilarityExampleSelector

입력과 의미적으로 가장 유사한 예제를 자동으로 선택합니다.


In [ ]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# ---------------------------------------------------
# SemanticSimilarityExampleSelector - 의미 유사도로 예제 자동 선택
# ---------------------------------------------------

# 1단계: SemanticSimilarityExampleSelector 생성
# SemanticSimilarityExampleSelector: 입력과 의미가 가장 비슷한 예제를 벡터 유사도로 자동 선택
#   - OpenAIEmbeddings: 텍스트를 벡터로 변환하는 임베딩 모델
#   - Chroma: 임베딩을 저장하고 유사도 검색을 수행하는 벡터 DB
#   - k=1: 가장 유사한 예제 1개만 선택 (토큰 절약)



In [ ]:
# ---------------------------------------------------
# 2단계: select_examples()로 유사 예제 검색 확인
# ---------------------------------------------------

# 2단계: 입력과 가장 유사한 예제 선택 테스트
# select_examples(): 입력 딕셔너리를 받아 k개의 예제를 반환
question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"



In [ ]:
# ---------------------------------------------------
# 3단계: Example Selector를 FewShotPromptTemplate에 적용
# ---------------------------------------------------

# 3단계: Example Selector를 FewShotPromptTemplate에 적용
# examples 대신 example_selector 사용 → 매 호출마다 입력에 맞는 예제를 동적으로 선택


# 체인 생성


# 실행



## 3. FewShotChatMessagePromptTemplate — 대화형 퓨샷 프롬프트

`FewShotChatMessagePromptTemplate`(퓨샷 채팅 메시지 프롬프트 템플릿)은 `ChatPromptTemplate`과 결합해 사용해요. 예제가 `(human, ai)` 메시지 쌍으로 표현되기 때문에 LLM이 역할 구분을 더 잘 이해할 수 있어요.

> **실무 팁:** 회의록 작성, 문서 요약, 문장 교정처럼 작업 유형이 여러 가지인 경우에는 `FewShotChatMessagePromptTemplate` + `ExampleSelector` 조합으로 입력에 맞는 예제를 자동으로 삽입하면 하나의 체인으로 멀티태스크를 처리할 수 있어요.

### 주요 특징

- 예제가 `("human", "{input}")` / `("ai", "{answer}")` 메시지 쌍으로 표현돼요
- `ChatPromptTemplate.from_messages()` 안에 `few_shot_prompt`를 그대로 넣을 수 있어요
- `ExampleSelector`와 결합하면 입력에 맞는 예제가 자동으로 삽입돼요

### 3.1 실용 예제 - 회의록/요약/교정 시스템

다양한 작업을 수행할 수 있는 멀티태스크 시스템을 구축합니다.


In [9]:
# 시나리오: instruction에 따라 다양한 작업(회의록, 요약, 교정)을 수행하는 시스템

# 1단계: 다양한 작업 유형의 예제 정의
task_examples = [
    {
        "instruction": "당신은 회의록 작성 전문가입니다. 주어진 정보를 바탕으로 회의록을 작성해 주세요",
        "input": "2023년 12월 25일, XYZ 회사의 마케팅 전략 회의가 오후 3시에 시작되었다. 회의에는 마케팅 팀장인 김수진, 디지털 마케팅 담당자인 박지민, 소셜 미디어 관리자인 이준호가 참석했다. 회의의 주요 목적은 2024년 상반기 마케팅 전략을 수립하고, 새로운 소셜 미디어 캠페인에 대한 아이디어를 논의하는 것이었다.",
        "answer": """회의록: XYZ 회사 마케팅 전략 회의
일시: 2023년 12월 25일 오후 3시
장소: XYZ 회사 회의실
참석자: 김수진 (마케팅 팀장), 박지민 (디지털 마케팅 담당자), 이준호 (소셜 미디어 관리자)

1. 개회
   - 회의 목적: 2024년 상반기 마케팅 전략 수립 및 새로운 소셜 미디어 캠페인 아이디어 논의

2. 주요 안건
   - 2024년 상반기 마케팅 전략 수립
   - 소셜 미디어 캠페인 기획

3. 마무리
   - 다음 회의 일정 확정 예정""",
    },
    {
        "instruction": "당신은 요약 전문가입니다. 다음 주어진 정보를 바탕으로 내용을 요약해 주세요",
        "input": "이 문서는 '지속 가능한 도시 개발을 위한 전략'에 대한 20페이지 분량의 보고서입니다. 보고서는 지속 가능한 도시 개발의 중요성, 현재 도시화의 문제점, 그리고 도시 개발을 지속 가능하게 만들기 위한 다양한 전략을 포괄적으로 다루고 있습니다. 이 보고서는 또한 성공적인 지속 가능한 도시 개발 사례를 여러 국가에서 소개하고 있습니다.",
        "answer": """문서 요약: 지속 가능한 도시 개발을 위한 전략 보고서

- 중요성: 지속 가능한 도시 개발이 필수적인 이유와 그에 따른 사회적, 경제적, 환경적 이익
- 현 문제점: 환경 오염, 자원 고갈, 불평등 증가 등 현재 도시화 과정의 주요 문제점
- 전략: 친환경 건축, 대중교통 개선, 에너지 효율성 증대, 지역사회 참여 강화
- 사례 연구: 덴마크 코펜하겐, 일본 요코하마 등 성공 사례
- 교훈: 다각적 접근의 중요성, 지역사회 협력, 장기적 계획의 필요성""",
    },
    {
        "instruction": "당신은 문장 교정 전문가입니다. 다음 주어진 문장을 교정해 주세요",
        "input": "우리 회사는 새로운 마케팅 전략을 도입하려고 한다. 이를 통해 고객과의 소통이 더 효과적이 될 것이다.",
        "answer": "본 회사는 새로운 마케팅 전략을 도입함으로써, 고객과의 소통을 보다 효과적으로 개선할 수 있을 것으로 기대된다.",
    },
]

print("=" * 60)
print("📚 준비된 작업 유형:", len(task_examples))
print("  1. 회의록 작성")
print("  2. 문서 요약")
print("  3. 문장 교정")
print("=" * 60)


📚 준비된 작업 유형: 3
  1. 회의록 작성
  2. 문서 요약
  3. 문장 교정


In [ ]:
# ---------------------------------------------------
# 2~4단계: ChatPromptTemplate 예제 포맷 + Selector + FewShot 구성
# ---------------------------------------------------

# 2단계: ChatPromptTemplate 형식의 예제 프롬프트 정의
# human/ai 쌍으로 예제를 구성하면 LLM이 역할 구분을 더 잘 이해함


# 3단계: 유사도 기반 Example Selector 생성


# 4단계: FewShotChatMessagePromptTemplate 생성
# FewShotChatMessagePromptTemplate: ChatPromptTemplate과 결합하여 대화형 퓨샷 프롬프트 구성
#   - example_selector: 입력에 맞는 예제를 자동으로 선택
#   - example_prompt: 각 예제를 human/ai 메시지 쌍으로 포맷



In [ ]:
# ---------------------------------------------------
# 5단계: 최종 ChatPromptTemplate 구성 및 체인 생성
# ---------------------------------------------------

# 5단계: 최종 ChatPromptTemplate 구성
# few_shot_prompt를 from_messages() 안에 직접 삽입 가능
# → 실행 시 입력에 맞는 예제가 자동으로 해당 위치에 삽입됨


# 체인 생성



In [ ]:
# 6단계: 테스트 - 회의록 작성
question = {
    "instruction": "회의록을 작성해 주세요",
    "input": "2023년 12월 26일, ABC 기술 회사의 제품 개발 팀은 새로운 모바일 애플리케이션 프로젝트에 대한 주간 진행 상황 회의를 가졌다. 이 회의에는 프로젝트 매니저인 최현수, 주요 개발자인 황지연, UI/UX 디자이너인 김태영이 참석했다. 회의의 주요 목적은 프로젝트의 현재 진행 상황을 검토하고, 다가오는 마일스톤에 대한 계획을 수립하는 것이었다.",
}



## 4. 커스텀 Example Selector 구현 — 내 기준으로 예제 선택하기

기본 `SemanticSimilarityExampleSelector`는 예제의 **모든 필드**를 합산해 유사도를 계산해요. 그러다 보면 `instruction`이 같아도 `input` 내용이 달라 엉뚱한 예제가 선택될 수 있어요.

**특정 필드만** 기준으로 삼고 싶을 때 `BaseExampleSelector`를 상속해서 직접 구현하면 돼요.

> **주의:** `BaseExampleSelector`를 상속할 때는 반드시 `select_examples`와 `add_example` 두 메서드를 구현해야 해요. 하나라도 빠지면 추상 클래스 오류가 발생합니다.

In [ ]:
# ---------------------------------------------------
# 문제 상황 재현 - 모든 필드를 합산한 유사도의 한계
# ---------------------------------------------------

# 문제 상황 재현: instruction만으로 검색
# task_selector는 examples의 모든 필드(instruction + input + answer)를 합산하여 임베딩
# → instruction만 입력하면 input 필드가 없어 유사도 계산이 부정확해질 수 있음



### 4.1 커스텀 Example Selector 클래스 구현

`instruction` 필드만을 사용하여 유사도를 계산하는 커스텀 Selector를 만듭니다.


In [14]:
from langchain_core.example_selectors.base import BaseExampleSelector
from typing import Dict, List
import numpy as np


class CustomExampleSelector(BaseExampleSelector):
    """instruction 필드만을 사용하여 유사도를 계산하는 커스텀 Example Selector
    
    BaseExampleSelector를 상속받아 select_examples 메서드를 구현합니다.
    """
    
    def __init__(self, examples: List[Dict], embeddings, k: int = 1):
        """커스텀 Example Selector 초기화
        
        Args:
            examples: 예제 리스트
            embeddings: 임베딩 모델 (OpenAIEmbeddings 등)
            k: 선택할 예제 개수 (기본값: 1)
        """
        self.examples = examples
        self.embeddings = embeddings
        self.k = k
        
        # 모든 예제의 instruction을 미리 임베딩하여 저장
        self.example_embeddings = self._embed_examples()
    
    def _embed_examples(self) -> List:
        """모든 예제의 instruction을 임베딩"""
        instructions = [example["instruction"] for example in self.examples]
        return self.embeddings.embed_documents(instructions)
    
    def _cosine_similarity(self, vec1: List[float], vec2: List[float]) -> float:
        """코사인 유사도 계산"""
        vec1 = np.array(vec1)
        vec2 = np.array(vec2)
        return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))
    
    def select_examples(self, input_variables: Dict[str, str]) -> List[Dict]:
        """입력과 가장 유사한 예제를 선택
        
        Args:
            input_variables: instruction 필드를 포함한 입력 딕셔너리
            
        Returns:
            가장 유사한 k개의 예제 리스트
        """
        # 입력의 instruction을 임베딩
        input_instruction = input_variables.get("instruction", "")
        input_embedding = self.embeddings.embed_query(input_instruction)
        
        # 각 예제와의 유사도 계산
        similarities = [
            self._cosine_similarity(input_embedding, example_emb)
            for example_emb in self.example_embeddings
        ]
        
        # 유사도가 높은 순으로 정렬하여 상위 k개 선택
        top_k_indices = np.argsort(similarities)[-self.k:][::-1]
        
        return [self.examples[i] for i in top_k_indices]
    
    def add_example(self, example: Dict) -> None:
        """새로운 예제 추가
        
        Args:
            example: 추가할 예제 딕셔너리
        """
        self.examples.append(example)
        # 새로운 예제의 instruction 임베딩 추가
        new_embedding = self.embeddings.embed_query(example["instruction"])
        self.example_embeddings.append(new_embedding)


print("=" * 60)
print("✅ CustomExampleSelector 클래스 정의 완료")
print("=" * 60)
print()
print("주요 메서드:")
print("  • __init__: 초기화 및 예제 임베딩")
print("  • select_examples: instruction 기반 유사 예제 선택")
print("  • add_example: 새로운 예제 추가")
print("=" * 60)


✅ CustomExampleSelector 클래스 정의 완료

주요 메서드:
  • __init__: 초기화 및 예제 임베딩
  • select_examples: instruction 기반 유사 예제 선택
  • add_example: 새로운 예제 추가


In [ ]:
# ---------------------------------------------------
# 1단계: 커스텀 Example Selector 인스턴스 생성
# ---------------------------------------------------

# 1단계: 커스텀 Example Selector 생성
# CustomExampleSelector: instruction 필드만 임베딩하여 유사도 계산



In [ ]:
# 2단계: instruction만으로 검색 테스트
test_cases = [
    {"instruction": "회의록을 작성해 주세요"},
    {"instruction": "다음 문서를 요약해 주세요"},
    {"instruction": "문장을 교정해 주세요"},
]



### 4.3 커스텀 Selector를 사용한 완전한 체인


In [ ]:
# ---------------------------------------------------
# 1~3단계: 커스텀 Selector로 완전한 체인 구성
# ---------------------------------------------------

# 1단계: 커스텀 Selector를 사용한 FewShotChatMessagePromptTemplate 생성


# 2단계: 최종 ChatPromptTemplate 구성


# 3단계: 체인 생성



In [ ]:
# 4단계: 최종 테스트
question = {
    "instruction": "회의록을 작성해 주세요",
    "input": "2024년 1월 10일, DEF 스타트업의 기술 팀 회의가 진행되었다. CTO인 박민수, 백엔드 개발자인 정수아, 프론트엔드 개발자인 김태준이 참석했다. 새로운 API 아키텍처 설계와 성능 최적화 방안에 대해 논의했다.",
}



## 연습 문제

직접 해보면서 학습 내용을 정리해봅시다!


### 연습 1: 코드 리뷰 Few-Shot 시스템

**과제**: 코드 리뷰를 수행하는 Few-Shot 시스템을 만드세요.

**요구사항**:
- `FewShotPromptTemplate` 사용
- 3개 이상의 코드 리뷰 예제 포함
- 리뷰 형식: 장점, 개선점, 제안사항
- Chain-of-Thought 패턴 적용


In [ ]:
# ============================================================
# TODO: FewShotPromptTemplate으로 코드 리뷰 Few-Shot 시스템을 만드세요
# 힌트: examples 리스트에 {"code": ..., "review": ...} 형식으로 3개 이상 예제를 정의하고
#       FewShotPromptTemplate(examples=..., example_prompt=..., suffix=..., input_variables=...)로 구성하세요
# 예상 결과: 입력 코드에 대해 "1단계 - 기능 파악 → 2단계 - 장점 → 3단계 - 개선점 → 4단계 - 제안사항" 형식의 리뷰
# ============================================================

# 여기에 코드를 작성하세요

# code_review_examples = [
#     {
#         "code": "...",
#         "review": "..."
#     },
#     ...
# ]

### 연습 1 풀이

In [ ]:
# 연습 1 풀이: 코드 리뷰 Few-Shot 시스템



### 연습 2: 다국어 번역 Example Selector

**과제**: 언어 쌍에 따라 적절한 번역 예제를 선택하는 시스템을 만드세요.

**요구사항**:
- `SemanticSimilarityExampleSelector` 사용
- 다양한 언어 쌍의 번역 예제 (한→영, 한→일, 영→한 등)
- 입력 언어 쌍에 맞는 예제 자동 선택


In [ ]:
# ============================================================
# TODO: SemanticSimilarityExampleSelector를 사용하여 다국어 번역 시스템을 만드세요
# 힌트: examples에 {"source_lang": ..., "target_lang": ..., "source": ..., "translation": ...} 형식으로
#       다양한 언어 쌍의 예제를 정의하고, SemanticSimilarityExampleSelector로 유사 예제를 자동 선택하세요
# 예상 결과: 입력 언어 쌍에 맞는 예제가 자동으로 선택되어 번역 결과 출력
# ============================================================

# 여기에 코드를 작성하세요

# translation_examples = [
#     {
#         "source_lang": "한국어",
#         "target_lang": "영어",
#         "source": "...",
#         "target": "..."
#     },
#     ...
# ]

### 연습 2 풀이

In [ ]:
# 연습 2 풀이: 다국어 번역 Example Selector



### 연습 3: 감정 분석 커스텀 Selector

**과제**: 감정 유형(긍정/부정/중립)에 따라 예제를 선택하는 커스텀 Selector를 만드세요.

**요구사항**:
- `BaseExampleSelector` 상속
- 감정 레이블 기반 정확한 매칭
- 같은 감정 유형의 예제만 선택


In [ ]:
# ============================================================
# TODO: BaseExampleSelector를 상속하여 감정 유형별 예제 선택기를 구현하세요
# 힌트: BaseExampleSelector를 상속하고 select_examples()와 add_example() 두 메서드를 반드시 구현해야 합니다
#       select_examples()에서 입력 텍스트의 키워드로 감정을 추정하고, 같은 감정 유형의 예제를 반환하세요
# 예상 결과: "정말 추천합니다" 입력 시 긍정 예제, "실망했습니다" 입력 시 부정 예제가 선택됨
# ============================================================

# 여기에 코드를 작성하세요

# class SentimentExampleSelector(BaseExampleSelector):
#     def __init__(self, examples, k=2):
#         ...
#     
#     def select_examples(self, input_variables):
#         ...

### 연습 3 풀이

In [ ]:
# 연습 3 풀이: 감정 분석 커스텀 Selector



## 마무리 — 이번 노트북에서 배운 것

| 개념 | 핵심 클래스 | 언제 쓰나요? |
|------|------------|-------------|
| 정적 퓨샷 | `FewShotPromptTemplate` | 예제 수가 적고 고정된 경우 |
| 동적 퓨샷 | `FewShotPromptTemplate` + `ExampleSelector` | 예제 풀이 크고 입력별로 선택이 필요한 경우 |
| 대화형 퓨샷 | `FewShotChatMessagePromptTemplate` | System/Human/AI 역할 구분이 필요한 경우 |
| 커스텀 선택 | `BaseExampleSelector` 상속 | 특정 필드 기준 또는 도메인 로직으로 예제를 선택할 때 |

퓨샷 기법은 프롬프트 엔지니어링에서 가장 효과가 검증된 방법 중 하나예요. 예제의 품질이 LLM 응답의 품질을 직접 결정하므로, 예제를 잘 고르는 것이 핵심이에요.

---

**다음 노트북 예고:** `03-ChatPromptTemplate` — 대화 이력을 동적으로 관리하는 `MessagesPlaceholder`(메시지 플레이스홀더)와 멀티턴(multi-turn) 챗봇 구현을 배워요.